# 实验一：多模态数据清洗与主键对齐

## HyCF-TFM 数据准备流水线

**目的**：解决原始实验中 `random.choice` 随机对齐的问题，通过构建基于主键 (`subject_id`, `hadm_id`) 的确定性关联，
将 Text（MIMIC-III + 结构化CSV）、ECG、Image 三种模态对齐到同一患者/住院记录，并输出标准化的 **Master Index CSV**。

### 对齐策略

| 模态 | 数据源 | 主键依据 | 对齐方式 |
|------|--------|----------|----------|
| Text | MIMIC-III (ADMISSIONS + NOTEEVENTS + DIAGNOSES_ICD + PATIENTS + ICUSTAYS) | `subject_id` / `hadm_id` | 原生主键关联 |
| Text | Heart_failure_clinical_records_dataset.csv | 合成 `subject_id` (HF_0001~) | 标签一致性对齐 |
| ECG  | PhysioNet2017 / CPSC2021 / CongestiveHF / FilesDB | 信号级分类标签 → 心衰风险标签映射 | 标签对齐 + 轮转分配 |
| Image | CheXchonet (metadata.csv with echocardiography labels) | `patient_id` + 心脏结构指标 | 心衰风险标签映射 |

### 输出
- `master_index.csv`：列名 `subject_id, hadm_id, text_path, ecg_path, image_path, label, label_type, data_source`
- 该文件为后续 Notebook 2（可视化）和 Notebook 3（模型训练）的唯一数据入口

In [ ]:
# ==============================
# Path Configuration
# ==============================
BASE = r"e:\平时作业\人工智能\data"

MIMIC_ROOT    = os.path.join(BASE, "mimic-iii-clinical-database-demo-1.4")
# Priority: use CareVue subset if available (contains real notes), else demo
MIMIC_CARE_ROOT = os.path.join(BASE, "mimic-iii-clinical-database-carevue-subset-1.4")
NOTEEVENTS_SRC = os.path.join(MIMIC_CARE_ROOT, "NOTEEVENTS.csv")
if not os.path.exists(NOTEEVENTS_SRC): # Try nested folder if consistent with unzip structure
    NOTEEVENTS_SRC = os.path.join(MIMIC_CARE_ROOT, "NOTEEVENTS.csv", "NOTEEVENTS.csv")
if not os.path.exists(NOTEEVENTS_SRC): 
    NOTEEVENTS_SRC = os.path.join(MIMIC_ROOT, "NOTEEVENTS.csv")

SUPP_CSV      = os.path.join(BASE, "Heart_failure_clinical_records_dataset.csv")
MIMIC_IV_ROOT = os.path.join(BASE, "mimic-iv-ext-cardiac-disease-1.0.0") # New Source

PHYSIONET_ECG = os.path.join(BASE, "af-classification-from-a-short-single-lead-ecg-recording-the-physionet-computing-in-cardiology-challenge-2017-1.0.0")
CPSC_ECG      = os.path.join(BASE, "paroxysmal-atrial-fibrillation-events-detection-from-dynamic-ecg-recordings-the-4th-china-physiological-signal-challenge-2021-1.0.0")
CHF_ECG       = os.path.join(BASE, "congestive-heart-failure-rr-interval-database-1.0.0")
FILES_ECG     = os.path.join(BASE, "files")
IMG_ROOT      = os.path.join(BASE, "images", "chexchonet-a-chest-radiograph-dataset-with-gold-standard-echocardiography-labels-1.0.0")

OUTPUT_CSV    = os.path.join(BASE, "master_index.csv")
TEXT_CACHE_DIR = os.path.join(BASE, "text_cache")
os.makedirs(TEXT_CACHE_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"Notes Source: {NOTEEVENTS_SRC}")
print(f"Output will be saved to: {OUTPUT_CSV}")

## Step 1: 解析 MIMIC-III 全部相关 CSV

从关系型数据库中重建每一次住院的完整临床画像：
- **ADMISSIONS.csv**: 住院基本信息、入院/出院时间、入院诊断
- **PATIENTS.csv**: 人口学（性别、年龄）
- **NOTEEVENTS.csv**: 出院小结（Discharge Summary）
- **DIAGNOSES_ICD.csv + D_ICD_DIAGNOSES.csv**: ICD-9 诊断码 → 可读诊断名
- **ICUSTAYS.csv**: ICU 住院时长
- **LABEVENTS.csv + D_LABITEMS.csv**: 关键实验室指标（BNP/Creatinine/Sodium）
- **PRESCRIPTIONS.csv**: 用药信息

In [ ]:
# ==============================
# 1A. ADMISSIONS — Base Table with Label
# ==============================
df_adm = pd.read_csv(os.path.join(MIMIC_ROOT, "ADMISSIONS.csv"))
df_adm['admittime'] = pd.to_datetime(df_adm['admittime'], errors='coerce')
df_adm['dischtime'] = pd.to_datetime(df_adm['dischtime'], errors='coerce')
df_adm = df_adm.sort_values(by=['subject_id', 'admittime'])

# 30-day Readmission Label
df_adm['next_admittime'] = df_adm.groupby('subject_id')['admittime'].shift(-1)
df_adm['gap_days'] = (df_adm['next_admittime'] - df_adm['dischtime']).dt.total_seconds() / 86400
df_adm['readmission_30d'] = ((df_adm['gap_days'] >= 0) & (df_adm['gap_days'] <= 30)).astype(int)

# In-hospital mortality (separate label)
df_adm['hospital_death'] = df_adm['hospital_expire_flag'].astype(int)

# Composite label: 1 if readmitted within 30d OR died in hospital
df_adm['adverse_outcome'] = ((df_adm['readmission_30d'] == 1) | (df_adm['hospital_death'] == 1)).astype(int)

print(f"ADMISSIONS: {len(df_adm)} records")
print(f"  Readmission 30d: {df_adm['readmission_30d'].sum()} positive")
print(f"  Hospital Death:  {df_adm['hospital_death'].sum()} positive")
print(f"  Adverse Outcome: {df_adm['adverse_outcome'].sum()} positive")
print(f"  Columns: {list(df_adm.columns[:10])}...")

In [ ]:
# ==============================
# 1B. PATIENTS — Demographics
# ==============================
df_pat = pd.read_csv(os.path.join(MIMIC_ROOT, "PATIENTS.csv"))
df_pat['dob'] = pd.to_datetime(df_pat['dob'], errors='coerce')

df_main = pd.merge(df_adm, df_pat[['subject_id', 'gender', 'dob', 'expire_flag']], on='subject_id', how='left')

# Safe Age Calculation (MIMIC shifts DOB 300 years for >89 patients)
df_main['age'] = df_main['admittime'].dt.year - df_main['dob'].dt.year
df_main['age'] = df_main['age'].apply(lambda x: 91.4 if (pd.notna(x) and x > 120) else (x if pd.notna(x) else 0))

print(f"After PATIENTS merge (MIMIC-III): {len(df_main)} records")

# --- MIMIC-IV Extension Integration ---
# Append valid admissions from the MIMIC-IV Heart Disease extension
linkage_path = os.path.join(MIMIC_IV_ROOT, "heart_diagnoses.csv")
if os.path.exists(linkage_path):
    print("Integrate MIMIC-IV Cohort...")
    try:
        df_iv_meta = pd.read_csv(linkage_path)
        if 'subject_id' in df_iv_meta.columns and 'hadm_id' in df_iv_meta.columns:
            # unique admissions
            df_iv_cohort = df_iv_meta[['subject_id', 'hadm_id']].drop_duplicates().copy()
            
            # Align columns: Initialize missing columns with default/NaN
            for col in df_main.columns:
                if col not in df_iv_cohort.columns:
                    df_iv_cohort[col] = np.nan
            
            # Set defaults for required fields
            df_iv_cohort['gender'] = 'Unknown' # Demographics not in linkage file
            df_iv_cohort['age'] = 65.0         # Placeholder age
            df_iv_cohort['short_title'] = 'MIMIC-IV Heart Failure' # Indicator
            
            # Append
            df_main = pd.concat([df_main, df_iv_cohort], ignore_index=True)
            print(f"  + Added {len(df_iv_cohort)} MIMIC-IV admissions.")
    except Exception as e:
        print(f"  Failed to load MIMIC-IV linkage: {e}")

print(f"Total Cohort Size: {len(df_main)} records")
print(f"  Age range: {df_main['age'].min():.0f} ~ {df_main['age'].max():.0f}")
print(f"  Gender: {dict(df_main['gender'].value_counts())}")

In [ ]:
# ==============================
# 1C. DIAGNOSES_ICD + D_ICD_DIAGNOSES — Readable Diagnosis List
# ==============================
df_diag = pd.read_csv(os.path.join(MIMIC_ROOT, "DIAGNOSES_ICD.csv"))
df_d_icd = pd.read_csv(os.path.join(MIMIC_ROOT, "D_ICD_DIAGNOSES.csv"))

df_diag_full = pd.merge(df_diag, df_d_icd[['icd9_code', 'short_title', 'long_title']], on='icd9_code', how='left')

# Heart Failure specific flag (ICD9: 428.x)
df_diag_full['is_hf'] = df_diag_full['icd9_code'].astype(str).str.startswith('428')

# Aggregate per admission
df_diag_agg = df_diag_full.groupby('hadm_id').agg(
    diagnosis_list=('long_title', lambda x: '; '.join(x.dropna().astype(str)[:15])),  # Top 15 diagnoses
    diagnosis_count=('icd9_code', 'count'),
    has_heart_failure=('is_hf', 'any')
).reset_index()

df_main = pd.merge(df_main, df_diag_agg, on='hadm_id', how='left')

print(f"DIAGNOSES: {len(df_diag_full)} total diagnosis entries")
print(f"  Heart Failure (428.x) related: {df_diag_full['is_hf'].sum()}")
print(f"  Admissions with HF diagnosis: {df_main['has_heart_failure'].sum()}")

In [ ]:
# ==============================
# 1D. NOTEEVENTS — Clinical Notes (CareVue + MIMIC-IV HPI)
# ==============================
import json

# --- Part 1: MIMIC-III CareVue Subset ---
print("Loading MIMIC-III CareVue Notes...")

# Handle Path Logic: Check if path is a directory containing the CSV
real_note_path = NOTEEVENTS_SRC
if os.path.isdir(real_note_path):
    print(f"  Path is a directory, looking inside: {real_note_path}")
    candidate = os.path.join(real_note_path, "NOTEEVENTS.csv")
    if os.path.exists(candidate):
        real_note_path = candidate
    else:
        print("  Error: NOTEEVENTS.csv not found inside directory.")

df_notes_iii = pd.DataFrame()
try:
    if os.path.exists(real_note_path) and os.path.isfile(real_note_path):
        print(f"  Reading: {real_note_path}")
        df_notes_iii = pd.read_csv(real_note_path, low_memory=False)
        df_notes_iii.columns = [c.lower() for c in df_notes_iii.columns]
        
        # Keep relevant columns
        if 'hadm_id' in df_notes_iii.columns and 'text' in df_notes_iii.columns:
            df_notes_iii = df_notes_iii[['hadm_id', 'category', 'text']]
            df_notes_iii['source'] = 'MIMIC-III'
            # Ensure hadm_id is numeric
            df_notes_iii['hadm_id'] = pd.to_numeric(df_notes_iii['hadm_id'], errors='coerce')
        else:
            print("  Warning: Standard columns (hadm_id, text) not found in CareVue CSV.")
            df_notes_iii = pd.DataFrame()
        print(f"  Loaded {len(df_notes_iii)} notes from MIMIC-III.")
    else:
        print(f"  File not found: {real_note_path}")
except Exception as e:
    print(f"  Error loading CareVue Notes: {e}")

# --- Part 2: MIMIC-IV Extension (HPI) ---
print("Loading MIMIC-IV HPI Data...")
hpi_json_path = os.path.join(MIMIC_IV_ROOT, "RAG_data", "HPI.json")
linkage_path = os.path.join(MIMIC_IV_ROOT, "heart_diagnoses.csv")

df_notes_iv = pd.DataFrame()

if os.path.exists(hpi_json_path) and os.path.exists(linkage_path):
    try:
        # 1. Load Linkage
        df_link = pd.read_csv(linkage_path)
        
        # 2. Load JSON Text
        with open(hpi_json_path, 'r', encoding='utf-8') as f:
            hpi_data = json.load(f)
        
        # Flatten and treat Keys as HADM_ID
        hpi_records = []
        for entry in hpi_data:
            for hid, text in entry.items():
                # Convert key to int if possible, assuming keys are hadm_id
                try:
                    h_id = int(hid)
                    hpi_records.append({'hadm_id': h_id, 'text': text})
                except:
                    continue
        
        df_hpi = pd.DataFrame(hpi_records)
        print(f"  Parsed {len(df_hpi)} HPI snippets from JSON.")
        
        if not df_hpi.empty:
            # 3. Merge with Linkage (Optional, but useful to confirm subject_id if needed later)
            # Actually, if we have hadm_id, that's the join key for df_main.
            # We can use df_link to filter for valid heart failure admissions if we wanted,
            # but for now let's just use the text.
            
            # Ensure types match for merge
            df_link['hadm_id'] = pd.to_numeric(df_link['hadm_id'], errors='coerce')
            df_hpi['hadm_id'] = pd.to_numeric(df_hpi['hadm_id'], errors='coerce')
            
            # Merge to filter or enrich
            # We use 'inner' to ensure we only keep notes for known admissions
            df_notes_iv = pd.merge(df_link[['hadm_id', 'subject_id']].drop_duplicates(), df_hpi, on='hadm_id', how='inner')
            
            # Format
            df_notes_iv['category'] = 'Discharge summary' 
            df_notes_iv['source'] = 'MIMIC-IV'
            df_notes_iv = df_notes_iv[['hadm_id', 'category', 'text', 'source']]
            
            print(f"  Loaded {len(df_notes_iv)} notes from MIMIC-IV (after linking).")

    except Exception as e:
        print(f"  Error processing HPI: {e}")

# --- Combine ---
# Ensure we have consistent columns
cols = ['hadm_id', 'category', 'text', 'source']
to_concat = []
if not df_notes_iii.empty: to_concat.append(df_notes_iii[cols])
if not df_notes_iv.empty: to_concat.append(df_notes_iv[cols])

if to_concat:
    df_notes = pd.concat(to_concat, ignore_index=True)
    # Clean
    df_notes = df_notes[df_notes['hadm_id'].notna()]
    df_notes['hadm_id'] = df_notes['hadm_id'].astype(int)

    # Filter Discharge summaries
    df_disch = df_notes[df_notes['category'] == 'Discharge summary'].copy()
    df_disch_agg = df_disch.groupby('hadm_id')['text'].apply(
        lambda x: ' '.join(x.dropna().astype(str))
    ).reset_index()
    df_disch_agg.rename(columns={'text': 'discharge_note'}, inplace=True)

    # Nursing notes
    df_nursing = df_notes[df_notes['category'].isin(['Nursing', 'Nursing/other'])].copy()
    if not df_nursing.empty:
        df_nursing_agg = df_nursing.groupby('hadm_id')['text'].apply(
            lambda x: ' '.join(x.dropna().astype(str)[:3])
        ).reset_index()
        df_nursing_agg.rename(columns={'text': 'nursing_note'}, inplace=True)
        # Merge nursing
        if 'nursing_note' in df_main.columns:
            df_main = df_main.drop(columns=['nursing_note'])
        df_main = pd.merge(df_main, df_nursing_agg, on='hadm_id', how='left')
    else:
        df_main['nursing_note'] = np.nan

    # Merge discharge
    if 'discharge_note' in df_main.columns:
        df_main = df_main.drop(columns=['discharge_note'])
    df_main = pd.merge(df_main, df_disch_agg, on='hadm_id', how='left')

    has_discharge = df_main['discharge_note'].notna().sum()
    has_nursing = df_main['nursing_note'].notna().sum()
    print(f"Total Notes Unified: {len(df_notes)}")
    print(f"  Admissions with Discharge Summary/HPI: {has_discharge}")
    print(f"  Admissions with Nursing Notes: {has_nursing}")
else:
    print("WARNING: No notes loaded from any source.")
    df_main['discharge_note'] = np.nan
    df_main['nursing_note'] = np.nan

In [ ]:
# ==============================
# 1E. ICUSTAYS — Length of Stay
# ==============================
df_icu = pd.read_csv(os.path.join(MIMIC_ROOT, "ICUSTAYS.csv"))
df_icu_agg = df_icu.groupby('hadm_id').agg(
    icu_los_total=('los', 'sum'),
    icu_stays_count=('icustay_id', 'count'),
    first_careunit=('first_careunit', 'first')
).reset_index()

df_main = pd.merge(df_main, df_icu_agg, on='hadm_id', how='left')
print(f"ICUSTAYS: {len(df_icu)} records")
print(f"  Mean ICU LOS: {df_icu['los'].mean():.2f} days")

In [ ]:
# ==============================
# 1F. LABEVENTS — Key Lab Values (Heart Failure markers)
# ==============================
df_lab = pd.read_csv(os.path.join(MIMIC_ROOT, "LABEVENTS.csv"))
df_labitems = pd.read_csv(os.path.join(MIMIC_ROOT, "D_LABITEMS.csv"))

# Key HF-related lab items
hf_lab_keywords = ['Creatinine', 'Sodium', 'Potassium', 'Troponin', 'BNP', 'Hemoglobin', 'Glucose', 'Lactate']
hf_items = df_labitems[df_labitems['label'].str.contains('|'.join(hf_lab_keywords), case=False, na=False)]
print(f"Identified {len(hf_items)} lab item types matching HF keywords")

df_lab_hf = pd.merge(df_lab, hf_items[['itemid', 'label']], on='itemid', how='inner')
df_lab_hf = df_lab_hf[df_lab_hf['valuenum'].notna()]

# Aggregate: Get last recorded value per admission per lab type
df_lab_hf['charttime'] = pd.to_datetime(df_lab_hf['charttime'], errors='coerce')
df_lab_last = df_lab_hf.sort_values('charttime').groupby(['hadm_id', 'label']).last().reset_index()

# Pivot to wide format
df_lab_pivot = df_lab_last.pivot_table(index='hadm_id', columns='label', values='valuenum', aggfunc='last')
df_lab_pivot.columns = ['lab_' + c.replace(' ', '_').lower() for c in df_lab_pivot.columns]
df_lab_pivot = df_lab_pivot.reset_index()

df_main = pd.merge(df_main, df_lab_pivot, on='hadm_id', how='left')
print(f"Lab columns added: {list(df_lab_pivot.columns[1:])}")

In [ ]:
# ==============================
# 1G. PRESCRIPTIONS — Key Medications
# ==============================
df_rx = pd.read_csv(os.path.join(MIMIC_ROOT, "PRESCRIPTIONS.csv"))

# HF-relevant drug classes
hf_drugs = ['Furosemide', 'Lisinopril', 'Metoprolol', 'Carvedilol', 'Spironolactone',
            'Digoxin', 'Warfarin', 'Heparin', 'Aspirin', 'Atorvastatin', 'Amiodarone']
pattern = '|'.join(hf_drugs)
df_rx_hf = df_rx[df_rx['drug'].str.contains(pattern, case=False, na=False)]

df_rx_agg = df_rx_hf.groupby('hadm_id')['drug'].apply(
    lambda x: '; '.join(sorted(set(x.dropna().astype(str))))
).reset_index()
df_rx_agg.rename(columns={'drug': 'hf_medications'}, inplace=True)

df_main = pd.merge(df_main, df_rx_agg, on='hadm_id', how='left')
print(f"PRESCRIPTIONS: {len(df_rx)} total, {len(df_rx_hf)} HF-related")
print(f"  Admissions with HF meds: {df_main['hf_medications'].notna().sum()}")

In [ ]:
# ==============================
# 1H. Construct Unified Text & Save to Cache
# ==============================
def build_clinical_text(row):
    """Compose a comprehensive clinical narrative from structured fields."""
    parts = []
    
    # Demographics
    gender = row.get('gender', 'Unknown')
    age = row.get('age', 0)
    parts.append(f"Patient: {gender}, {age:.0f} years old.")
    
    # Admission context
    if pd.notna(row.get('admission_type')):
        parts.append(f"Admission type: {row['admission_type']}.")
    if pd.notna(row.get('diagnosis')):
        parts.append(f"Primary admission diagnosis: {row['diagnosis']}.")
    
    # ICU
    if pd.notna(row.get('icu_los_total')):
        parts.append(f"ICU stay: {row['icu_los_total']:.1f} days in {row.get('first_careunit', 'ICU')}.")
    
    # Diagnoses (ICD)
    if pd.notna(row.get('diagnosis_list')) and len(str(row['diagnosis_list'])) > 5:
        parts.append(f"Diagnoses: {str(row['diagnosis_list'])[:600]}")
    
    # Lab values
    lab_cols = [c for c in row.index if c.startswith('lab_')]
    lab_parts = []
    for lc in lab_cols:
        if pd.notna(row[lc]):
            lab_parts.append(f"{lc.replace('lab_', '').title()}: {row[lc]:.2f}")
    if lab_parts:
        parts.append(f"Lab results: {', '.join(lab_parts)}.")
    
    # Medications
    if pd.notna(row.get('hf_medications')):
        parts.append(f"Medications: {row['hf_medications']}.")
    
    # Discharge note (truncated)
    if pd.notna(row.get('discharge_note')) and len(str(row['discharge_note'])) > 10:
        clean = str(row['discharge_note']).replace('\n', ' ').replace('\r', ' ')
        parts.append(f"Discharge summary: {clean[:1500]}")
    elif pd.notna(row.get('nursing_note')) and len(str(row['nursing_note'])) > 10:
        clean = str(row['nursing_note']).replace('\n', ' ').replace('\r', ' ')
        parts.append(f"Nursing note: {clean[:800]}")
    
    return ' '.join(parts)

# Build and save text for each admission
mimic_records = []
for idx, row in df_main.iterrows():
    # Must have at least some textual or diagnostic info
    has_text = pd.notna(row.get('discharge_note')) or pd.notna(row.get('nursing_note'))
    has_diag = pd.notna(row.get('diagnosis_list')) and len(str(row.get('diagnosis_list', ''))) > 5
    if not has_text and not has_diag:
        continue
    
    text = build_clinical_text(row)
    
    # Save text to file (deterministic filename based on hadm_id)
    try:
        hadm_id = int(row['hadm_id'])
        subj_id = int(row['subject_id'])
    except ValueError:
        continue # Skip if IDs are invalid
        
    txt_filename = f"mimic_{subj_id}_{hadm_id}.txt"
    txt_path = os.path.join(TEXT_CACHE_DIR, txt_filename)
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(text)
    
    # Handle missing labels (MIMIC-IV extension might not have labels yet, default to 0/Unknown)
    label_val = row.get('adverse_outcome')
    label_readm = row.get('readmission_30d')
    label_mort = row.get('hospital_death')

    mimic_records.append({
        'subject_id': f"MIMIC_{subj_id}",
        'hadm_id': hadm_id,
        'text_path': txt_path,
        'label': int(label_val) if pd.notna(label_val) else 0,
        'label_readmission': int(label_readm) if pd.notna(label_readm) else 0,
        'label_mortality': int(label_mort) if pd.notna(label_mort) else 0,
        'label_type': 'readmission+mortality',
        'data_source': 'MIMIC-III' if row.get('source') != 'MIMIC-IV' else 'MIMIC-IV',
        'age': row.get('age', 0),
        'gender': row.get('gender', 'Unknown'),
        'has_hf_diagnosis': bool(row.get('has_heart_failure', False)),
        'has_discharge_note': pd.notna(row.get('discharge_note')),
        'diagnosis_primary': str(row.get('diagnosis', ''))[:100]
    })

df_mimic_index = pd.DataFrame(mimic_records)
print(f"\nMIMIC-III/IV Text Index: {len(df_mimic_index)} valid admissions")
if not df_mimic_index.empty:
    print(f"  Label distribution: {dict(df_mimic_index['label'].value_counts())}")
    print(f"  Data Source distribution: {dict(df_mimic_index['data_source'].value_counts())}")
    print(f"  With HF diagnosis: {df_mimic_index['has_hf_diagnosis'].sum()}")
    print(f"  With discharge note: {df_mimic_index['has_discharge_note'].sum()}")

## Step 2: 解析补充 CSV（Heart Failure Dataset）

将结构化表格转化为自然语言文本，赋予合成 `subject_id`。

In [ ]:
# ==============================
# 2. Supplementary CSV → Text
# ==============================
df_supp = pd.read_csv(SUPP_CSV)
print(f"Supplementary CSV: {len(df_supp)} records, columns: {list(df_supp.columns)}")

supp_records = []
for i, row in df_supp.iterrows():
    # Build clinical narrative
    parts = []
    sex_str = 'Male' if row.get('sex', 0) == 1 else 'Female'
    parts.append(f"Patient: {sex_str}, {row['age']:.0f} years old.")
    parts.append(f"Follow-up period: {row.get('time', 'N/A')} days.")
    
    conditions = []
    if row.get('anaemia', 0) == 1: conditions.append('Anaemia')
    if row.get('diabetes', 0) == 1: conditions.append('Diabetes')
    if row.get('high_blood_pressure', 0) == 1: conditions.append('Hypertension')
    if row.get('smoking', 0) == 1: conditions.append('Smoking')
    if conditions:
        parts.append(f"Comorbidities: {', '.join(conditions)}.")
    else:
        parts.append("No major comorbidities.")
    
    parts.append(f"Ejection Fraction: {row.get('ejection_fraction', 'N/A')}%.")
    parts.append(f"Serum Creatinine: {row.get('serum_creatinine', 'N/A')} mg/dL.")
    parts.append(f"Serum Sodium: {row.get('serum_sodium', 'N/A')} mEq/L.")
    parts.append(f"CPK: {row.get('creatinine_phosphokinase', 'N/A')} mcg/L.")
    parts.append(f"Platelets: {row.get('platelets', 'N/A')} kiloplatelets/mL.")
    parts.append("Primary Diagnosis: Heart Failure.")
    
    text = ' '.join(parts)
    
    subj_id = f"HF_{i:04d}"
    txt_filename = f"supp_{subj_id}.txt"
    txt_path = os.path.join(TEXT_CACHE_DIR, txt_filename)
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(text)
    
    supp_records.append({
        'subject_id': subj_id,
        'hadm_id': -1,  # No hadm_id for external data
        'text_path': txt_path,
        'label': int(row.get('DEATH_EVENT', 0)),
        'label_readmission': -1,  # Not applicable
        'label_mortality': int(row.get('DEATH_EVENT', 0)),
        'label_type': 'mortality',
        'data_source': 'SuppCSV',
        'age': row['age'],
        'gender': sex_str[0],  # M or F
        'has_hf_diagnosis': True,
        'has_discharge_note': False,
        'diagnosis_primary': 'Heart Failure'
    })

df_supp_index = pd.DataFrame(supp_records)
print(f"\nSupplementary Text Index: {len(df_supp_index)} records")
print(f"  Label distribution: {dict(df_supp_index['label'].value_counts())}")

## Step 3: 构建 ECG 索引（带标签映射）

- **PhysioNet2017**: 使用 REFERENCE-v3.csv, 标签 A(AF)=高风险, N(Normal)=低风险, O/~=中等
- **CPSC2021**: 所有样本标记为心律异常风险
- **CongestiveHF**: 全部为充血性心衰患者 → label=1
- **FilesDB**: 额外心电 → 按文件名推断

In [ ]:
# ==============================
# 3. ECG Index with Labels
# ==============================
def scan_ecg_with_signal_check(folder, source_name):
    """Scan for .hea files with matching signal files."""
    results = []
    if not os.path.exists(folder):
        print(f"  [SKIP] {source_name}: path not found")
        return results
    headers = glob.glob(os.path.join(folder, '**', '*.hea'), recursive=True)
    for h in headers:
        base = os.path.splitext(h)[0]
        if any(os.path.exists(base + ext) for ext in ['.mat', '.dat', '.ecg']):
            results.append(base)
    print(f"  {source_name}: {len(results)} valid ECGs")
    return results

print("Scanning ECG Sources...")
ecg_records = []

# A. PhysioNet 2017 (with reference labels)
physionet_ecgs = scan_ecg_with_signal_check(PHYSIONET_ECG, "PhysioNet2017")
ref_path = os.path.join(PHYSIONET_ECG, "REFERENCE-v3.csv")
ecg_labels_pn = {}
if os.path.exists(ref_path):
    df_ref = pd.read_csv(ref_path, header=None, names=['record', 'label'])
    # A=AF(high risk), N=Normal(low risk), O=Other(medium), ~=Noisy(exclude)
    label_map = {'A': 1, 'N': 0, 'O': 0, '~': -1}
    ecg_labels_pn = dict(zip(df_ref['record'], df_ref['label'].map(label_map)))

for ecg_path in physionet_ecgs:
    rec_name = os.path.basename(ecg_path)
    lbl = ecg_labels_pn.get(rec_name, 0)
    if lbl == -1: continue  # Skip noisy
    ecg_records.append({
        'ecg_path': ecg_path,
        'ecg_label': lbl,
        'ecg_source': 'PhysioNet2017'
    })

# B. CPSC 2021
cpsc_ecgs = scan_ecg_with_signal_check(CPSC_ECG, "CPSC2021")
for ecg_path in cpsc_ecgs:
    ecg_records.append({'ecg_path': ecg_path, 'ecg_label': 1, 'ecg_source': 'CPSC2021'})

# C. Congestive Heart Failure DB (ALL are CHF patients → label=1)
chf_ecgs = scan_ecg_with_signal_check(CHF_ECG, "CongestiveHF")
for ecg_path in chf_ecgs:
    ecg_records.append({'ecg_path': ecg_path, 'ecg_label': 1, 'ecg_source': 'CongestiveHF'})

# D. FilesDB
files_ecgs = scan_ecg_with_signal_check(FILES_ECG, "FilesDB")
for ecg_path in files_ecgs:
    ecg_records.append({'ecg_path': ecg_path, 'ecg_label': 0, 'ecg_source': 'FilesDB'})

df_ecg_index = pd.DataFrame(ecg_records)
print(f"\nTotal ECG Index: {len(df_ecg_index)}")
print(f"  By source: {dict(df_ecg_index['ecg_source'].value_counts())}")
print(f"  Label dist: {dict(df_ecg_index['ecg_label'].value_counts())}")

## Step 4: 构建 Image 索引（CheXchonet metadata）

CheXchonet 提供了心超标签 (`lvidd`, `ivsd`, `slvh`, `dlv`)，可以派生心衰风险标签：
- **LVIDd ≥ 5.6 cm** (男) / **≥ 5.2 cm** (女) → 左室扩大 → 高风险
- **composite_slvh_dlv = 1** → 心室肥厚/扩大 → 高风险

In [ ]:
# ==============================
# 4. Image Index with cardiac labels
# ==============================
metadata_path = os.path.join(IMG_ROOT, "metadata.csv")
df_img_meta = pd.read_csv(metadata_path)
print(f"CheXchonet metadata: {len(df_img_meta)} records")
print(f"  Columns: {list(df_img_meta.columns)}")

# Derive Heart Failure risk label from echocardiography
# LVIDd threshold: >5.6cm (M) or >5.2cm (F) indicates dilated LV
# composite_slvh_dlv: 1 = structural heart disease
def derive_cardiac_risk(row):
    # Structural abnormality flag
    if row.get('composite_slvh_dlv', 0) == 1:
        return 1
    # LVIDd thresholds
    lvidd = row.get('lvidd', None)
    sex = row.get('sex', 'M')
    if pd.notna(lvidd):
        threshold = 5.6 if sex == 'M' else 5.2
        if lvidd > threshold:
            return 1
    return 0

df_img_meta['cardiac_risk'] = df_img_meta.apply(derive_cardiac_risk, axis=1)

# Build full image paths
img_dir = os.path.join(IMG_ROOT, "images")
image_records = []
for _, row in df_img_meta.iterrows():
    fname = row['cxr_filename']
    full_path = os.path.join(img_dir, fname)
    if os.path.exists(full_path):
        image_records.append({
            'image_path': full_path,
            'image_patient_id': row['patient_id'],
            'image_label': row['cardiac_risk'],
            'image_age': row.get('age', None),
            'image_sex': row.get('sex', None),
            'lvidd': row.get('lvidd', None),
            'ef_proxy': row.get('composite_slvh_dlv', 0)
        })

df_img_index = pd.DataFrame(image_records)
print(f"\nImage Index: {len(df_img_index)} valid images")
print(f"  Cardiac risk dist: {dict(df_img_index['image_label'].value_counts())}")

## Step 5: 主键对齐 — 基于标签一致性的确定性分配

**核心策略**：由于 MIMIC/SuppCSV 与 ECG/Image 来自不同患者群体，不存在天然主键。
我们采用 **标签一致 + 人口学匹配 + 轮转分配** 的确定性对齐方案（非随机）：

1. Text 样本按 `label` 分组 → `{0: [...], 1: [...]}`
2. ECG 按 `ecg_label` 分组 → `{0: [...], 1: [...]}`  
3. Image 按 `image_label` 分组 → `{0: [...], 1: [...]}`
4. 对于每个 Text 样本（主键方），按标签从对应池中 **轮转取出** ECG 和 Image 路径

这保证了：
- 同一行数据的 text label ≈ ecg label ≈ image label（标签一致性）
- 完全确定性（相同数据 → 相同对齐结果）
- 无 `random.choice`

In [ ]:
# ==============================
# 5. Deterministic Label-Consistent Alignment
# ==============================

# Combine MIMIC + Supp text records
df_text_all = pd.concat([df_mimic_index, df_supp_index], ignore_index=True)
print(f"Combined Text Pool: {len(df_text_all)}")
print(f"  Label dist: {dict(df_text_all['label'].value_counts())}")

# Group ECGs by label
ecg_pool = {0: [], 1: []}
for _, row in df_ecg_index.iterrows():
    ecg_pool[row['ecg_label']].append(row['ecg_path'])

# Group Images by label
img_pool = {0: [], 1: []}
for _, row in df_img_index.iterrows():
    img_pool[row['image_label']].append(row['image_path'])

print(f"\nECG Pool: label_0={len(ecg_pool[0])}, label_1={len(ecg_pool[1])}")
print(f"Image Pool: label_0={len(img_pool[0])}, label_1={len(img_pool[1])}")

# --- Deterministic Round-Robin Assignment ---
ecg_counters = {0: 0, 1: 0}
img_counters = {0: 0, 1: 0}

master_rows = []
alignment_stats = {'full_triple': 0, 'text_ecg': 0, 'text_img': 0, 'text_only': 0}

for _, text_row in df_text_all.iterrows():
    label = text_row['label']
    
    # Assign ECG (round-robin within same label group)
    ecg_path = None
    if ecg_pool[label]:
        ecg_idx = ecg_counters[label] % len(ecg_pool[label])
        ecg_path = ecg_pool[label][ecg_idx]
        ecg_counters[label] += 1
    elif ecg_pool[1 - label]:  # Fallback: use other label group
        ecg_idx = ecg_counters[1 - label] % len(ecg_pool[1 - label])
        ecg_path = ecg_pool[1 - label][ecg_idx]
        ecg_counters[1 - label] += 1
    
    # Assign Image (round-robin within same label group)
    image_path = None
    if img_pool[label]:
        img_idx = img_counters[label] % len(img_pool[label])
        image_path = img_pool[label][img_idx]
        img_counters[label] += 1
    elif img_pool[1 - label]:  # Fallback
        img_idx = img_counters[1 - label] % len(img_pool[1 - label])
        image_path = img_pool[1 - label][img_idx]
        img_counters[1 - label] += 1
    
    # Track modality coverage
    has_ecg = ecg_path is not None
    has_img = image_path is not None
    if has_ecg and has_img:
        alignment_stats['full_triple'] += 1
    elif has_ecg:
        alignment_stats['text_ecg'] += 1
    elif has_img:
        alignment_stats['text_img'] += 1
    else:
        alignment_stats['text_only'] += 1
    
    master_rows.append({
        'subject_id': text_row['subject_id'],
        'hadm_id': text_row['hadm_id'],
        'text_path': text_row['text_path'],
        'ecg_path': ecg_path if ecg_path else '',
        'image_path': image_path if image_path else '',
        'label': label,
        'label_readmission': text_row['label_readmission'],
        'label_mortality': text_row['label_mortality'],
        'label_type': text_row['label_type'],
        'data_source': text_row['data_source'],
        'age': text_row['age'],
        'gender': text_row['gender'],
        'has_ecg': has_ecg,
        'has_image': has_img,
        'has_text': True
    })

df_master = pd.DataFrame(master_rows)

print(f"\n{'='*50}")
print(f"MASTER INDEX BUILT: {len(df_master)} samples")
print(f"{'='*50}")
print(f"Label distribution: {dict(df_master['label'].value_counts())}")
print(f"Data source: {dict(df_master['data_source'].value_counts())}")
print(f"\nModality Coverage:")
print(f"  Text + ECG + Image (Full Triple): {alignment_stats['full_triple']}")
print(f"  Text + ECG only:                  {alignment_stats['text_ecg']}")
print(f"  Text + Image only:                {alignment_stats['text_img']}")
print(f"  Text only:                        {alignment_stats['text_only']}")

In [ ]:
# ==============================
# 6. Save Master Index CSV
# ==============================
df_master.to_csv(OUTPUT_CSV, index=False)
print(f"Master Index saved to: {OUTPUT_CSV}")
print(f"File size: {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB")
print(f"\nPreview:")
df_master.head(10)

In [ ]:
# ==============================
# 7. Verification: Read back and validate
# ==============================
df_verify = pd.read_csv(OUTPUT_CSV)

print("=== Master Index Verification ===")
print(f"Total samples: {len(df_verify)}")
print(f"Columns: {list(df_verify.columns)}")
print(f"\nLabel Distribution:")
print(df_verify['label'].value_counts())
print(f"\nModality Coverage:")
print(f"  Has Text:  {df_verify['has_text'].sum()} / {len(df_verify)} ({df_verify['has_text'].mean()*100:.1f}%)")
print(f"  Has ECG:   {df_verify['has_ecg'].sum()} / {len(df_verify)} ({df_verify['has_ecg'].mean()*100:.1f}%)")
print(f"  Has Image: {df_verify['has_image'].sum()} / {len(df_verify)} ({df_verify['has_image'].mean()*100:.1f}%)")
triple_pct = ((df_verify['has_text']) & (df_verify['has_ecg']) & (df_verify['has_image'])).mean() * 100
print(f"  Full Triple: {triple_pct:.1f}%")

# Spot check: verify text files exist
sample_paths = df_verify['text_path'].head(5).tolist()
print(f"\nText file spot check:")
for p in sample_paths:
    exists = os.path.exists(p)
    size = os.path.getsize(p) if exists else 0
    print(f"  {os.path.basename(p)}: {'OK' if exists else 'MISSING'} ({size} bytes)")